# SARIMAX baselines (daily seasonality + exogenous)

ERCOT prices have strong daily (m=24) and weekly seasonality. We run two SARIMAX baselines:

- **A.** Univariate `auto_arima` (pmdarima) with `m=24` to pick (p,d,q)(P,D,Q,24).
- **B.** Fixed-order SARIMAX with the same curated exogenous regressors as Prophet.

These should be compared against the XGBoost baseline in `02_v2_models.ipynb` (referenced by name; we don't reload its outputs here).

In [ ]:
# Cell 1 - setup
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from models.utils import (
    chronological_split, select_enhanced_features,
    TARGET_REG, TARGET_CLASS,
    regression_report, classification_report,
    apply_feature_transforms,
)
FEATURES = PROJECT_ROOT / "data" / "features"


In [ ]:
# Cell 2 - load & split
mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)
features = select_enhanced_features(mat)
print(f"{len(features)} features  train={len(train):,}  val={len(val):,}  test={len(test):,}")


In [ ]:
!pip install pmdarima --quiet

## Section A — Univariate auto_arima baseline

Stepwise search on the full 60k+ row train set is too slow. We subsample to the **last 2 years of training data (2021–2022)** for the order search, then forecast 2024 with `dynamic=False` (the model uses true past values up to each forecast origin and forecasts 1 step forward). This mirrors the operational use case where each hour's prediction sees the previous hour's realised price.

In [ ]:
from pmdarima import auto_arima

df_reg = mat.dropna(subset=[TARGET_REG]).sort_index()
tr_reg, vl_reg, te_reg = chronological_split(df_reg)

# Subsample training data: last 2 years of train (2021-2022) for faster stepwise search.
y_train_full = tr_reg[TARGET_REG]
y_train_sub = y_train_full[y_train_full.index >= "2021-01-01"]
y_test = te_reg[TARGET_REG]
print(f"auto_arima search rows: {len(y_train_sub):,}  test rows: {len(y_test):,}")

auto_model = auto_arima(
    y_train_sub.values,
    seasonal=True, m=24,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    trace=True,
    max_p=3, max_q=3, max_P=2, max_Q=2,
    n_jobs=1,
)
print(auto_model.summary())


In [ ]:
# Forecast 2024 with dynamic=False: refit-free in-sample-style forecasts where the model conditions
# on observed history up to each forecast origin. Implementation: append the unseen segment between
# the search window and the test window, then use predict_in_sample over the test indices.
import statsmodels.api as sm

# Refit equivalent statsmodels SARIMAX on (train+val up to test_start) using the auto_arima order
# so we can use the dynamic=False predict path cleanly.
order = auto_model.order
seasonal_order = auto_model.seasonal_order
print(f"selected order={order} seasonal_order={seasonal_order}")

y_fit = pd.concat([tr_reg[TARGET_REG], vl_reg[TARGET_REG]])
y_full = pd.concat([y_fit, y_test])

mod_uni = sm.tsa.statespace.SARIMAX(
    y_full.values,
    order=order, seasonal_order=seasonal_order,
    enforce_stationarity=False, enforce_invertibility=False,
)
# Fit parameters on the train+val portion only by passing nobs via apply on fitted params,
# but simpler: fit on y_fit, then apply the params to y_full and predict over the test slice.
res_fit = sm.tsa.statespace.SARIMAX(
    y_fit.values,
    order=order, seasonal_order=seasonal_order,
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)
res_applied = mod_uni.smooth(res_fit.params)

start = len(y_fit)
end = len(y_full) - 1
preds_uni = res_applied.predict(start=start, end=end, dynamic=False)
preds_uni = pd.Series(preds_uni, index=y_test.index, name="sarimax-univariate")

uni_report = regression_report(y_test.values, preds_uni.values, name="sarimax-univariate-test")
uni_report


## Section B — SARIMAX with exogenous regressors

Fixed order `(2,0,2)(1,0,1,24)` with the same 10-regressor set used in `13_prophet.ipynb`. Fit on train+val, forecast on 2024 with `dynamic=False`.

In [ ]:
REGRESSORS = ['hubavg_lag_1h', 'hubavg_lag_24h', 'hubavg_lag_168h', 'temp_max_across_zones', 'wind_mean_across_zones', 'load_actual_mw', 'henry_hub_price', 'da_HB_HUBAVG', 'hour', 'is_peak_hours']
missing = [c for c in REGRESSORS if c not in mat.columns]
assert not missing, f"Missing regressors: {missing}"


In [ ]:
# Build aligned exog frames over the same indices as y_fit / y_test.
needed = [TARGET_REG] + REGRESSORS
df_x = mat[needed].dropna().sort_index()
tr_x, vl_x, te_x = chronological_split(df_x)

y_fit_x = pd.concat([tr_x[TARGET_REG], vl_x[TARGET_REG]])
y_test_x = te_x[TARGET_REG]
X_fit = pd.concat([tr_x[REGRESSORS], vl_x[REGRESSORS]]).values
X_test = te_x[REGRESSORS].values
y_full_x = pd.concat([y_fit_x, y_test_x])
X_full = np.vstack([X_fit, X_test])

print(f"fit rows: {len(y_fit_x):,}  test rows: {len(y_test_x):,}  exog cols: {X_fit.shape[1]}")


In [ ]:
# Fit SARIMAX(2,0,2)(1,0,1,24) with exogenous regressors. NOTE: this is slow on full history;
# if it OOMs locally, the user will run on the GPU server with more RAM.
res_x_fit = sm.tsa.statespace.SARIMAX(
    y_fit_x.values, exog=X_fit,
    order=(2, 0, 2), seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False, maxiter=50)
print(res_x_fit.summary())

mod_x_full = sm.tsa.statespace.SARIMAX(
    y_full_x.values, exog=X_full,
    order=(2, 0, 2), seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False, enforce_invertibility=False,
)
res_x_applied = mod_x_full.smooth(res_x_fit.params)

start_x = len(y_fit_x)
end_x = len(y_full_x) - 1
preds_x = res_x_applied.predict(start=start_x, end=end_x, dynamic=False)
preds_x = pd.Series(preds_x, index=y_test_x.index, name="sarimax-exogenous")

exog_report = regression_report(y_test_x.values, preds_x.values, name="sarimax-exogenous-test")
exog_report


## Section C — Comparison

Side-by-side metrics for both SARIMAX variants. Compare against the XGBoost baseline in `02_v2_models.ipynb` (not reloaded here).

In [ ]:
comparison = pd.DataFrame([uni_report, exog_report]).set_index("name")
print(comparison.to_string(float_format=lambda x: f"{x:.4f}"))


## Local execution blocker

Authored but not executed end-to-end on the local `.venv` (Python 3.9). Section A's `pmdarima` does not install cleanly here (no compatible wheel for this Python/numpy combo). Section B's statsmodels SARIMAX is available locally but a SARIMAX(2,0,2)(1,0,1,24) fit on ~70k rows with 10 exogenous regressors is RAM- and time-prohibitive on this machine. **Run this notebook on the GPU server** (more RAM, working pmdarima).

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np

# Variant 1: SARIMAX univariate
_y_true_dollar = np.asarray(y_test).reshape(-1)
_y_pred_dollar = np.asarray(preds_uni).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC [sarimax-univariate]: {_spike_pr_auc:.3f}")

# Variant 2: SARIMAX with exogenous regressors
_y_true_x_dollar = np.asarray(y_test_x).reshape(-1)
_y_pred_x_dollar = np.asarray(preds_x).reshape(-1)
_spike_pr_auc_x = average_precision_score(
    (_y_true_x_dollar > 200).astype(int),
    _y_pred_x_dollar,
)
print(f"  Spike PR-AUC [sarimax-exogenous]:  {_spike_pr_auc_x:.3f}")